In [ ]:
def Overlap_hist_sin_fit(histograms, bins, filter_value, omega=False):

    seconds_per_year = 365.25 * 24 * 3600

    h_avg = ROOT.TH1D("h_avg_temp", "", bins, 0, seconds_per_year)
    h_avg.SetDirectory(0)

    counts = np.zeros(bins)
    n_used = np.zeros(bins)   # number of values contributing to each bin

    for _, h in histograms:

        year_seconds = h.GetXaxis().GetXmax()

        for i in range(1, bins+1):

            value = h.GetBinContent(i)

            if value < filter_value:
                continue  

            frac = h.GetBinCenter(i) / year_seconds
            x = frac * seconds_per_year
            j = h_avg.FindBin(x)

            counts[j-1] += value
            n_used[j-1] += 1

    for i in range(bins):
        if n_used[i] > 0:
            counts[i] /= n_used[i]

    for i in range(1, bins+1):
        h_avg.SetBinContent(i, counts[i-1])

    xmin = h_avg.GetXaxis().GetXmin()
    xmax = h_avg.GetXaxis().GetXmax()

    fit = ROOT.TF1("global_sin_temp", sin_fit, xmin, xmax, 4)
    fit.SetNpx(5000)

    offset_guess = h_avg.Integral() / h_avg.GetNbinsX()
    amplitude_guess = 0.1 * offset_guess
    omega_guess = 2*np.pi / seconds_per_year

    fit.SetParameters(amplitude_guess, omega_guess, 0, offset_guess)

    if omega:
        fit.FixParameter(1, omega_guess)

    h_avg.Fit(fit, "W0")

    fit.SetLineColor(ROOT.kBlack)
    fit.SetLineWidth(2)
    fit.Draw("SAME")

    return fit, h_avg

In [ ]:
#Bromsgrove using year_hist looped 

histograms = []
bins = 180

for i,year in enumerate(Bromsgrove_year_files):
    colour = colours[i]
    h = year_hist(year, Bromsgrove_year_files[year], UNIX_year_start, colour, bins)
    histograms.append((year, h))

ymax = 1.2 * max(h.GetMaximum() for year, h in histograms)
c_all = ROOT.TCanvas("c_all_years", "All Years", 900, 600)

histograms[0][1].SetTitle("Bromsgrove Overlayed Data")

first = True
for year, h in histograms:
    h.SetMaximum(ymax)
    h.SetMinimum(0)

    if first:
        #h.Draw("P")
        first = False
    #else:
        #h.Draw("P SAME")

leg = ROOT.TLegend(0.1, 0.82, 0.9, 0.9)
leg.SetNColumns(len(histograms))

for year, h in histograms:
    leg.AddEntry(h, year, "l")

leg.Draw()
filter_value = 4*10**-3
fit, h_avg = Overlap_hist_sin_fit(histograms, bins, filter_value)

c_all.Draw()

In [ ]:
#UoB using year_hist_alongside with sin fit

offset = 0
histograms = []
bins = 500

for i, year in enumerate(UoB_year_files):
    colour = colours[i]
    hist = year_hist_alongside(year, UoB_year_files[year], UNIX_year_start, colour, offset, UoB_total_span, bins)
    histograms.append((year, hist))
    offset += UoB_year_lengths[year]

ymax = 1.2 * max(h.GetMaximum() for _, h in histograms)


c_side = ROOT.TCanvas("c_side_by_side", "Years Side By Side", 1100, 500)

histograms[0][1].SetTitle("University of Birmingham Sine Fit")

first = True
for year, h in histograms:
    h.SetMaximum(ymax)
    h.SetMinimum(0)

    if first:
        h.Draw("P")
        first = False
    else:
        h.Draw("P SAME")


leg = ROOT.TLegend(0.1, 0.82, 0.9, 0.9)
leg.SetNColumns(len(histograms))

for year, h in histograms:
    leg.AddEntry(h, year, "l")

leg.Draw()

fix_omega = True
filter_value = 0.015
fit = hist_sin_fit(histograms, UoB_total_span, bins, filter_value, fix_omega)

c_side.Draw()

In [ ]:
#Bromsgrove using year_hist_alongside with sin fit

offset = 0
histograms = []
bins = 200


for i, year in enumerate(Bromsgrove_year_files):
    colour = colours[i]
    hist = year_hist_alongside(year, Bromsgrove_year_files[year], UNIX_year_start, colour, offset, Bromsgrove_total_span, bins)
    histograms.append((year, hist))
    offset += Bromsgrove_year_lengths[year]

ymax = 1.2 * max(h.GetMaximum() for _, h in histograms)


c_side = ROOT.TCanvas("c_side_by_side", "Years Side By Side", 1100, 500)

histograms[0][1].SetTitle("Bromsgrove Data")

first = True
for year, h in histograms:
    h.SetMaximum(ymax)
    h.SetMinimum(0)

    if first:
        h.Draw("P")
        first = False
    else:
        h.Draw("P SAME")


leg = ROOT.TLegend(0.1, 0.82, 0.9, 0.9)
leg.SetNColumns(len(histograms))

for year, h in histograms:
    leg.AddEntry(h, year, "l")

leg.Draw()

fix_omega = False
filter_value = 23e-3
fit = hist_sin_fit(histograms, Bromsgrove_total_span, bins, filter_value, fix_omega)

c_side.Draw()